# LeetCode 2231 — Largest Number After Digit Swaps by Parity bằng Python

Link đề: https://leetcode.com/problems/largest-number-after-digit-swaps-by-parity/ · Độ khó: **Easy** · Chủ đề: Sorting, Heap (Priority Queue)

Skeleton LeetCode:

```python
class Solution:
    def largestInteger(self, num: int) -> int:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu **parity** (tính chẵn lẻ) của chữ số và cách tách số nguyên thành danh sách chữ số.
2. Nhận ra rằng "hoán đổi các chữ số **cùng tính chẵn lẻ**" nghĩa là **hai nhóm độc lập**: nhóm chẵn và nhóm lẻ, mỗi nhóm có thể xếp lại tùy ý.
3. Biết cách tạo số lớn nhất: đặt chữ số lớn hơn vào vị trí **cao hơn** (bên trái) trong từng nhóm.
4. Viết 4–5 cách: hoán đổi tham lam kiểu chọn (selection), sắp xếp từng nhóm, max-heap, và đếm theo bucket `0..9`.
5. Trace, test ngẫu nhiên đối chiếu, và tránh lỗi như sắp xếp toàn bộ chữ số, hoặc trộn lẫn hai nhóm parity.

## 2. Đề bài & ví dụ

Cho số nguyên dương `num`. Bạn có thể **hoán đổi hai chữ số bất kỳ có cùng tính chẵn lẻ** (cùng chẵn hoặc cùng lẻ) với số lần tùy ý. Trả về **giá trị lớn nhất** có thể đạt được.

| Ví dụ | num | Output | Giải thích |
|---|---|---|---|
| 1 | `1234` | `3412` | đổi `1↔3` (cùng lẻ) → `3214`; đổi `2↔4` (cùng chẵn) → `3412` |
| 2 | `65875` | `87655` | lẻ `{5,5,7}` xếp `7,5,5`; chẵn `{6,8}` xếp `8,6`: `8 7 6 5 5` |

Constraints: `1 <= num <= 10^9`.

Chú ý: ta **không** được đổi chỗ chữ số chẵn với chữ số lẻ. Vị trí nào đang giữ chữ số chẵn thì sau cùng vẫn giữ một chữ số chẵn (có thể khác).

In [1]:
examples = [(1234, 3412), (65875, 87655)]
for num, expected in examples:
    digits = [int(c) for c in str(num)]
    print(num, "| chữ số:", digits, "| mong đợi:", expected)

1234 | chữ số: [1, 2, 3, 4] | mong đợi: 3412
65875 | chữ số: [6, 5, 8, 7, 5] | mong đợi: 87655


## 3. Kiến thức nền

### 3.1. Tách chữ số: `str(num)` và `int(c)`

`str(1234)` là `"1234"`; duyệt từng ký tự và đổi `int(c)` cho ra danh sách chữ số `[1, 2, 3, 4]`. Ngược lại, ghép chữ số thành số bằng `int("".join(map(str, digits)))`.

In [2]:
num = 65875
digits = [int(c) for c in str(num)]
print(digits)
print(int("".join(map(str, digits))))
print(list(map(int, str(num))))

[6, 5, 8, 7, 5]
65875
[6, 5, 8, 7, 5]


### 3.2. Parity: `d % 2`

`d % 2 == 0` nghĩa là `d` chẵn (`0, 2, 4, 6, 8`); `d % 2 == 1` nghĩa là lẻ (`1, 3, 5, 7, 9`). Ta dùng nó để chia mỗi chữ số vào **nhóm chẵn** hoặc **nhóm lẻ**.

In [3]:
digits = [6, 5, 8, 7, 5]
evens = [d for d in digits if d % 2 == 0]
odds = [d for d in digits if d % 2 == 1]
print("chẵn:", evens, "| lẻ:", odds)
print([d % 2 for d in digits])

chẵn: [6, 8] | lẻ: [5, 7, 5]
[0, 1, 0, 1, 1]


### 3.3. `sorted(..., reverse=True)` và `list.sort`

`sorted(a)` trả về danh sách mới xếp tăng dần; `reverse=True` xếp giảm dần. `a.sort()` sắp xếp tại chỗ. Muốn lấy từng phần tử "lớn nhất còn lại" ta dùng `pop(0)` trên danh sách giảm dần, hoặc `iter(...)` / `next(...)`.

In [4]:
a = [6, 8]
print(sorted(a), sorted(a, reverse=True))
it = iter(sorted([5, 7, 5], reverse=True))
print(next(it), next(it), next(it))

[6, 8] [8, 6]
7 5 5


### 3.4. `heapq` — hàng đợi ưu tiên (min-heap)

`heapq` lấy ra phần tử **nhỏ nhất** bằng `heappop`. Muốn lấy phần tử **lớn nhất**, đẩy vào **số âm** (`-d`) rồi đổi dấu khi lấy ra. `heapify(list)` biến list thành heap trong `O(n)`; mỗi `heappop` tốn `O(log n)`.

In [5]:
import heapq
h = [-d for d in [5, 7, 5]]
heapq.heapify(h)
print([-heapq.heappop(h) for _ in range(3)])   # lớn nhất trước

[7, 5, 5]


### 3.5. Mảng đếm (bucket) theo chữ số `0..9`

Các chữ số chỉ có 10 giá trị. Mảng `cnt[d]` ghi số lần chữ số `d` xuất hiện; duyệt `d` từ `9` xuống `0` ta lấy được các chữ số **theo thứ tự giảm dần** mà không cần sắp xếp so sánh.

In [6]:
digits = [6, 5, 8, 7, 5]
cnt = [0] * 10
for d in digits:
    cnt[d] += 1
print(cnt)
print([d for d in range(9, -1, -1) for _ in range(cnt[d])])

[0, 0, 0, 0, 0, 2, 1, 1, 1, 0]
[8, 7, 6, 5, 5]


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Lấy `num = 65875`, các chữ số theo vị trí (từ trái): `6 5 8 7 5`. Ghi parity: `C L C L L` (C = chẵn, L = lẻ).

- Các vị trí **chẵn** là vị trí 0 và 2, chứa `{6, 8}`.
- Các vị trí **lẻ** là vị trí 1, 3, 4, chứa `{5, 7, 5}`.

Vì được hoán đổi tùy ý **trong cùng nhóm**, ta có thể đặt `8` (lớn nhất nhóm chẵn) vào vị trí chẵn **đầu tiên** (vị trí 0), rồi `6` vào vị trí chẵn tiếp theo (vị trí 2). Nhóm lẻ: `7` vào vị trí 1, rồi `5, 5` vào vị trí 3 và 4. Kết quả `8 7 6 5 5 = 87655`.

Quy tắc tạo số lớn nhất: chữ số ở **vị trí cao hơn** có trọng số lớn hơn (`10^k`), nên đặt chữ số lớn nhất của nhóm vào vị trí cao nhất mà nhóm đó chiếm.

In [7]:
digits = [6, 5, 8, 7, 5]
pos_even = [i for i, d in enumerate(digits) if d % 2 == 0]
pos_odd = [i for i, d in enumerate(digits) if d % 2 == 1]
print("vị trí giữ chữ số chẵn:", pos_even, "| vị trí giữ chữ số lẻ:", pos_odd)
print("chẵn giảm dần:", sorted((digits[i] for i in pos_even), reverse=True))
print("lẻ giảm dần  :", sorted((digits[i] for i in pos_odd), reverse=True))

vị trí giữ chữ số chẵn: [0, 2] | vị trí giữ chữ số lẻ: [1, 3, 4]
chẵn giảm dần: [8, 6]
lẻ giảm dần  : [7, 5, 5]


## 5. Cách 1 — Brute Force / baseline: hoán đổi tham lam kiểu chọn (selection)

Với mỗi cặp vị trí `i < j` có **cùng parity**, nếu `digits[j] > digits[i]` thì hoán đổi. Duyệt hết mọi cặp một lần giống thuật toán sắp xếp chọn (selection sort) nhưng chỉ cho cặp cùng parity: kết quả mỗi nhóm được xếp giảm dần.

In [8]:
def largest_swap_pairs(num):
    d = [int(c) for c in str(num)]
    n = len(d)
    for i in range(n):
        for j in range(i + 1, n):
            if d[i] % 2 == d[j] % 2 and d[j] > d[i]:
                d[i], d[j] = d[j], d[i]
    return int("".join(map(str, d)))

print(largest_swap_pairs(1234), largest_swap_pairs(65875))

3412 87655


In [9]:
from typing import List

class SolutionBrute:
    def largestInteger(self, num: int) -> int:
        d = [int(c) for c in str(num)]
        n = len(d)
        for i in range(n):
            for j in range(i + 1, n):
                if d[i] % 2 == d[j] % 2 and d[j] > d[i]:
                    d[i], d[j] = d[j], d[i]
        return int("".join(map(str, d)))

print(SolutionBrute().largestInteger(1234))
print(SolutionBrute().largestInteger(65875))

3412
87655


**Độ phức tạp:** Gọi `n` là số chữ số (`n ≤ 10`). Hai vòng lồng nhau → **Time `O(n²)`**, **Space `O(n)`** (danh sách chữ số).

**Nút thắt:** ta so sánh mọi cặp. Dù `n ≤ 10` nên rất nhỏ, ý tưởng "tách hai nhóm rồi sắp xếp" gọn hơn và dễ chứng minh hơn.

## 6. Ý tưởng thuật toán / bước đột phá

Insight: hoán đổi cùng parity khiến **tập vị trí chẵn** và **tập vị trí lẻ** độc lập. Bài toán tách thành hai bài nhỏ:

- nhóm **chẵn**: các vị trí `P_E`, các chữ số `E` → gán chữ số lớn nhất của `E` vào vị trí trái nhất của `P_E`, rồi lớn nhì vào vị trí kế...
- nhóm **lẻ**: làm tương tự với `P_O` và `O`.

Quy tắc: **nếu vị trí `i` đang giữ chữ số chẵn thì lấy chữ số chẵn lớn nhất còn lại; nếu lẻ thì lấy chữ số lẻ lớn nhất còn lại.** Cách này tối ưu vì ở vị trí trái nhất, chữ số lớn hơn luôn cho giá trị tổng lớn hơn bất kể phần còn lại.

## 7. Các cách tiếp theo

### Cách 2 — Sắp xếp hai nhóm giảm dần rồi "điền lại"

Tách `evens` và `odds`, sắp xếp mỗi nhóm giảm dần, rồi duyệt các chữ số gốc: nếu chữ số đó chẵn thì lấy `evens.pop(0)`, lẻ thì lấy `odds.pop(0)`. Cách rõ ràng và dễ giải thích nhất.

In [10]:
from typing import List

class SolutionSort:
    def largestInteger(self, num: int) -> int:
        digits = [int(c) for c in str(num)]
        evens = sorted((d for d in digits if d % 2 == 0), reverse=True)
        odds = sorted((d for d in digits if d % 2 == 1), reverse=True)
        res = []
        ei = oi = 0
        for d in digits:
            if d % 2 == 0:
                res.append(evens[ei]); ei += 1
            else:
                res.append(odds[oi]); oi += 1
        return int("".join(map(str, res)))

print(SolutionSort().largestInteger(1234))
print(SolutionSort().largestInteger(65875))

3412
87655


**Độ phức tạp:** Time `O(n log n)` (sắp xếp), Space `O(n)`. Dùng chỉ số `ei, oi` thay cho `pop(0)` để tránh tốn `O(n)` mỗi lần.

### Cách 3 — Hai max-heap

Đẩy chữ số chẵn vào `heap_even`, chữ số lẻ vào `heap_odd` (dạng số âm). Duyệt các chữ số gốc; chữ số chẵn → lấy đỉnh `heap_even`, lẻ → lấy đỉnh `heap_odd`. Thích hợp khi nhận chữ số theo dòng hoặc `n` rất lớn.

In [11]:
import heapq
from typing import List

class SolutionHeap:
    def largestInteger(self, num: int) -> int:
        digits = [int(c) for c in str(num)]
        heap_even = [-d for d in digits if d % 2 == 0]
        heap_odd = [-d for d in digits if d % 2 == 1]
        heapq.heapify(heap_even)
        heapq.heapify(heap_odd)
        res = []
        for d in digits:
            if d % 2 == 0:
                res.append(-heapq.heappop(heap_even))
            else:
                res.append(-heapq.heappop(heap_odd))
        return int("".join(map(str, res)))

print(SolutionHeap().largestInteger(1234))
print(SolutionHeap().largestInteger(65875))

3412
87655


**Độ phức tạp:** Time `O(n log n)` (mỗi `heappop` tốn `O(log n)`), Space `O(n)`.

### Cách 4 — Mảng đếm bucket với con trỏ theo parity

Đếm số lần mỗi chữ số xuất hiện. Với mỗi vị trí, ta cần "chữ số lớn nhất còn lại cùng parity". Duy trì hai con trỏ `max_even` (bắt đầu 8) và `max_odd` (bắt đầu 9); khi `cnt[max_even] == 0` thì giảm `max_even -= 2`. Không dùng sắp xếp so sánh.

In [12]:
from typing import List

class SolutionBucket:
    def largestInteger(self, num: int) -> int:
        digits = [int(c) for c in str(num)]
        cnt = [0] * 10
        for d in digits:
            cnt[d] += 1
        me, mo = 8, 9
        ans = 0
        for d in digits:
            if d % 2 == 0:
                while cnt[me] == 0:
                    me -= 2
                cnt[me] -= 1
                ans = ans * 10 + me
            else:
                while cnt[mo] == 0:
                    mo -= 2
                cnt[mo] -= 1
                ans = ans * 10 + mo
        return ans

print(SolutionBucket().largestInteger(1234))
print(SolutionBucket().largestInteger(65875))

3412
87655


**Độ phức tạp:** Time `O(n + 10) = O(n)` (hai con trỏ chỉ đi xuống tối đa 5 bước mỗi nhóm), Space `O(10) = O(1)`. Ta còn xây số trực tiếp `ans = ans * 10 + d` nên không cần ghép chuỗi.

### Cách 5 (mở rộng) — Sắp xếp một lần với khóa `(parity, -digit)`

Một thủ thuật gọn: sắp xếp **mọi chữ số** theo `(d % 2, -d)` được `[chẵn giảm dần..., lẻ giảm dần...]`. Rồi cắt hai nửa và lấy lần lượt bằng iterator.

In [13]:
from typing import List

class SolutionKey:
    def largestInteger(self, num: int) -> int:
        digits = [int(c) for c in str(num)]
        ordered = sorted(digits, key=lambda d: (d % 2, -d))
        k = sum(1 for d in digits if d % 2 == 0)
        it_even, it_odd = iter(ordered[:k]), iter(ordered[k:])
        res = [next(it_even) if d % 2 == 0 else next(it_odd) for d in digits]
        return int("".join(map(str, res)))

print(sorted([6, 5, 8, 7, 5], key=lambda d: (d % 2, -d)))
print(SolutionKey().largestInteger(1234))
print(SolutionKey().largestInteger(65875))

[8, 6, 7, 5, 5]
3412
87655


**Độ phức tạp:** Time `O(n log n)`, Space `O(n)`. Chủ yếu để học cách dùng `key=` với tuple.

## 8. TRACE từng bước

Trace Cách 2 với `num = 65875`:

| vị trí | chữ số gốc | parity | lấy từ nhóm | `evens` / `odds` còn lại | chữ số đặt |
|---|---|---|---|---|---|
| 0 | 6 | chẵn | `evens = [8, 6]` | `[6]` | **8** |
| 1 | 5 | lẻ | `odds = [7, 5, 5]` | `[5, 5]` | **7** |
| 2 | 8 | chẵn | `evens = [6]` | `[]` | **6** |
| 3 | 7 | lẻ | `odds = [5, 5]` | `[5]` | **5** |
| 4 | 5 | lẻ | `odds = [5]` | `[]` | **5** |

Kết quả `8 7 6 5 5 = 87655`.

In [14]:
def trace_sort(num):
    digits = [int(c) for c in str(num)]
    evens = sorted((d for d in digits if d % 2 == 0), reverse=True)
    odds = sorted((d for d in digits if d % 2 == 1), reverse=True)
    print("chữ số:", digits, "| evens:", evens, "| odds:", odds)
    res = []
    ei = oi = 0
    for i, d in enumerate(digits):
        if d % 2 == 0:
            put = evens[ei]; ei += 1; grp = "chẵn"
        else:
            put = odds[oi]; oi += 1; grp = "lẻ"
        res.append(put)
        print(f"  vị trí {i}: gốc={d} ({grp}) -> đặt {put} | res={res}")
    return int("".join(map(str, res)))

print("kết quả:", trace_sort(65875))
print("kết quả:", trace_sort(1234))

chữ số: [6, 5, 8, 7, 5] | evens: [8, 6] | odds: [7, 5, 5]
  vị trí 0: gốc=6 (chẵn) -> đặt 8 | res=[8]
  vị trí 1: gốc=5 (lẻ) -> đặt 7 | res=[8, 7]
  vị trí 2: gốc=8 (chẵn) -> đặt 6 | res=[8, 7, 6]
  vị trí 3: gốc=7 (lẻ) -> đặt 5 | res=[8, 7, 6, 5]
  vị trí 4: gốc=5 (lẻ) -> đặt 5 | res=[8, 7, 6, 5, 5]
kết quả: 87655
chữ số: [1, 2, 3, 4] | evens: [4, 2] | odds: [3, 1]
  vị trí 0: gốc=1 (lẻ) -> đặt 3 | res=[3]
  vị trí 1: gốc=2 (chẵn) -> đặt 4 | res=[3, 4]
  vị trí 2: gốc=3 (lẻ) -> đặt 1 | res=[3, 4, 1]
  vị trí 3: gốc=4 (chẵn) -> đặt 2 | res=[3, 4, 1, 2]
kết quả: 3412


In [15]:
def trace_bucket(num):
    digits = [int(c) for c in str(num)]
    cnt = [0] * 10
    for d in digits:
        cnt[d] += 1
    print("cnt =", cnt)
    me, mo, ans = 8, 9, 0
    for i, d in enumerate(digits):
        if d % 2 == 0:
            while cnt[me] == 0:
                me -= 2
            cnt[me] -= 1; put = me
        else:
            while cnt[mo] == 0:
                mo -= 2
            cnt[mo] -= 1; put = mo
        ans = ans * 10 + put
        print(f"  vị trí {i}: gốc={d} -> đặt {put}, ans={ans}, me={me}, mo={mo}")
    return ans

trace_bucket(65875)

cnt = [0, 0, 0, 0, 0, 2, 1, 1, 1, 0]
  vị trí 0: gốc=6 -> đặt 8, ans=8, me=8, mo=9
  vị trí 1: gốc=5 -> đặt 7, ans=87, me=8, mo=7
  vị trí 2: gốc=8 -> đặt 6, ans=876, me=6, mo=7
  vị trí 3: gốc=7 -> đặt 5, ans=8765, me=6, mo=5
  vị trí 4: gốc=5 -> đặt 5, ans=87655, me=6, mo=5


87655

## 9. Code nộp LeetCode

In [16]:
from typing import List

class Solution:
    def largestInteger(self, num: int) -> int:
        digits = [int(c) for c in str(num)]
        evens = sorted((d for d in digits if d % 2 == 0), reverse=True)
        odds = sorted((d for d in digits if d % 2 == 1), reverse=True)
        ei = oi = 0
        ans = 0
        for d in digits:
            if d % 2 == 0:
                ans = ans * 10 + evens[ei]
                ei += 1
            else:
                ans = ans * 10 + odds[oi]
                oi += 1
        return ans

print(Solution().largestInteger(1234))
print(Solution().largestInteger(65875))

3412
87655


## 10. Vì sao thuật toán đúng

- **Hai nhóm độc lập:** hoán đổi chỉ xảy ra giữa hai chữ số cùng parity nên **tập vị trí chẵn** (vị trí đang giữ chữ số chẵn) không thay đổi; mọi hoán vị của các chữ số chẵn lên các vị trí chẵn đều đạt được (hoán đổi sinh ra mọi hoán vị).
- **Tối ưu (so sánh từ trái sang phải):** hai số cùng độ dài so sánh theo chữ số khác nhau đầu tiên từ trái. Tại vị trí trái nhất, chữ số lớn nhất có thể đặt là phần tử lớn nhất còn lại của nhóm cùng parity. Chọn nó là tối ưu vì mọi lựa chọn nhỏ hơn cho số nhỏ hơn bất kể phần sau (**chọn tham lam đúng**).
- **Invariant của thuật toán:** sau khi xử lý `i` vị trí đầu, ta đã dùng đúng `i` chữ số lớn nhất theo thứ tự có thể của từng nhóm; phần còn lại của mỗi nhóm giữ nguyên là các chữ số nhỏ hơn.
- **Kết luận:** mỗi nhóm được xếp giảm dần trên các vị trí của nó, tương ứng số lớn nhất.

## 11. Phân tích Time / Space Complexity

Đặt `n` = số chữ số của `num` (`n ≤ 10`, vì `num ≤ 10^9`).

| Cách | Time | Space |
|---|---|---|
| 1. Hoán đổi mọi cặp | `O(n²)` | `O(n)` |
| 2. Sắp xếp hai nhóm | `O(n log n)` | `O(n)` |
| 3. Hai max-heap | `O(n log n)` | `O(n)` |
| 4. Bucket đếm `0..9` | `O(n)` | `O(1)` |
| 5. Sắp xếp một lần với khóa tuple | `O(n log n)` | `O(n)` |

Với `n ≤ 10`, mọi cách là hằng số nhỏ. Về lý thuyết cách bucket `O(n)` là nhanh nhất, nhưng cách 2 dễ viết và dễ đọc nhất.

## 12. Bộ test

Ví dụ đề, edge case (một chữ số, tất cả chẵn, tất cả lẻ, đã tối ưu sẵn), và test ngẫu nhiên đối chiếu **tất cả cách** với một lời giải brute force độc lập (thử mọi hoán vị cùng parity).

In [17]:
import random
from itertools import permutations

solvers = [SolutionBrute(), SolutionSort(), SolutionHeap(), SolutionBucket(), SolutionKey(), Solution()]

fixed = [
    (1234, 3412),
    (65875, 87655),
    (7, 7),                       # một chữ số
    (2468, 8642),                 # toàn chẵn -> giảm dần
    (1357, 7531),                 # toàn lẻ -> giảm dần
    (1000000000, 1000000000),     # 10^9: chẵn/lẻ xen kẽ sẵn
    (12, 12),                     # lẻ trước chẵn, không đổi được
    (21, 21),
    (1020, 1200),                 # chẵn {0,2,0} ở vị trí 1,2,3 -> 2,0,0
]
for num, expected in fixed:
    for sol in solvers:
        assert sol.largestInteger(num) == expected, (type(sol).__name__, num, sol.largestInteger(num), expected)
print("test cố định: OK")

def oracle(num):
    d = [int(c) for c in str(num)]
    pe = [i for i, x in enumerate(d) if x % 2 == 0]
    po = [i for i, x in enumerate(d) if x % 2 == 1]
    best = 0
    ev = [d[i] for i in pe]; od = [d[i] for i in po]
    for pe_perm in set(permutations(ev)):
        for po_perm in set(permutations(od)):
            t = d[:]
            for i, x in zip(pe, pe_perm): t[i] = x
            for i, x in zip(po, po_perm): t[i] = x
            best = max(best, int("".join(map(str, t))))
    return best

random.seed(2231)
for _ in range(300):
    num = random.randint(1, 10 ** 6)
    expected = oracle(num)
    for sol in solvers:
        assert sol.largestInteger(num) == expected, (type(sol).__name__, num)
print("300 test ngẫu nhiên đối chiếu oracle: OK")
for _ in range(300):
    num = random.randint(1, 10 ** 9)
    vals = {sol.largestInteger(num) for sol in solvers}
    assert len(vals) == 1, (num, vals)
print("300 test ngẫu nhiên đến 10^9: OK")

test cố định: OK
300 test ngẫu nhiên đối chiếu oracle: OK
300 test ngẫu nhiên đến 10^9: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Sắp xếp **toàn bộ** chữ số giảm dần

Cách này bỏ qua ràng buộc parity và có thể cho số lớn hơn mức hợp lệ.

In [18]:
def wrong1(num):
    return int("".join(sorted(str(num), reverse=True)))

print("sai :", wrong1(1234), "(4321 không đạt được vì phải đổi `1` lẻ với `4` chẵn)")
print("đúng:", Solution().largestInteger(1234))

sai : 4321 (4321 không đạt được vì phải đổi `1` lẻ với `4` chẵn)
đúng: 3412


### Lỗi 2: Trộn nhóm chẵn và lẻ khi điền lại

Nếu chỉ sắp xếp hai nhóm rồi nối nhóm chẵn rồi nhóm lẻ (không theo vị trí parity gốc), vị trí parity bị phá vỡ.

In [19]:
def wrong2(num):
    d = [int(c) for c in str(num)]
    evens = sorted((x for x in d if x % 2 == 0), reverse=True)
    odds = sorted((x for x in d if x % 2 == 1), reverse=True)
    return int("".join(map(str, odds + evens)))

print("sai :", wrong2(1234), "| đúng:", Solution().largestInteger(1234))

sai : 3142 | đúng: 3412


### Lỗi 3: Sắp xếp tăng dần thay vì giảm dần

Muốn số **lớn nhất** thì chữ số lớn ở vị trí cao (trái). Quên `reverse=True` cho ra số nhỏ nhất.

In [20]:
def wrong3(num):
    d = [int(c) for c in str(num)]
    evens = sorted(x for x in d if x % 2 == 0)         # thiếu reverse=True
    odds = sorted(x for x in d if x % 2 == 1)
    ei = oi = 0
    out = []
    for x in d:
        if x % 2 == 0:
            out.append(evens[ei]); ei += 1
        else:
            out.append(odds[oi]); oi += 1
    return int("".join(map(str, out)))

print("sai :", wrong3(65875), "| đúng:", Solution().largestInteger(65875))

sai : 65857 | đúng: 87655


### Lỗi 4: Dùng `pop(0)` hoặc `remove` làm mất thứ tự / chậm

`list.pop(0)` đúng nhưng `O(n)` mỗi lần. Dùng chỉ số `ei, oi` hoặc `deque`/`iter`. Một lỗi nữa: `digits.remove(x)` xóa phần tử **đầu tiên bằng `x`**, có thể xóa nhầm chỗ.

In [21]:
a = [5, 7, 5]
a.remove(5)
print(a, "<- remove(5) xóa lần xuất hiện ĐẦU TIÊN")
b = [7, 5, 5]
print(b.pop(0), b)

[7, 5] <- remove(5) xóa lần xuất hiện ĐẦU TIÊN
7 [5, 5]


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Thấy "hoán đổi các phần tử **thỏa tính chất** (cùng parity, cùng nhóm...)" → các nhóm là **lớp tương đương**; xử lý **từng nhóm riêng** rồi gộp lại theo vị trí.
- "Lớn nhất / nhỏ nhất sau nhiều lần hoán đổi tự do" → **sắp xếp** nhóm (giảm dần cho lớn nhất, tăng dần cho nhỏ nhất).
- Chữ số chỉ có 10 giá trị → **đếm bucket** nhanh hơn sắp xếp.
- Ghép lại thành số: `ans = ans * 10 + d` tránh ghép chuỗi.
- Với số có ít chữ số (`≤ 10`) đừng ngại viết cách đơn giản nhất; ưu tiên đúng và dễ đọc.

## 15. Biến thể của bài

1. **Nhỏ nhất** thay vì lớn nhất (sắp xếp tăng dần mỗi nhóm; chú ý số `0` đầu).
2. Cho phép chỉ **tối đa `k` lần** hoán đổi (cần xét lại; liên quan LeetCode 670 Maximum Swap).
3. Phân nhóm theo tiêu chí khác (chia 3 dư mấy, chữ số nguyên tố/không).
4. Hoán đổi các **chữ cái** trong một xâu theo nhóm nguyên âm / phụ âm.
5. LeetCode 2164 Sort Even and Odd Indices Independently: hai nhóm theo **chỉ số** chẵn/lẻ.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Hoán đổi mọi cặp | selection sort có điều kiện parity | `O(n²)` | `O(n)` | n nhỏ, dễ nghĩ ra |
| 2. Sắp xếp hai nhóm | tách, sort, điền lại theo parity | `O(n log n)` | `O(n)` | **khuyến nghị khi nộp** |
| 3. Hai max-heap | lấy đỉnh theo parity | `O(n log n)` | `O(n)` | dữ liệu theo dòng, cần luyện heap |
| 4. Bucket `0..9` | đếm + con trỏ giảm dần | `O(n)` | `O(1)` | nhanh nhất, chữ số ít giá trị |
| 5. Sort một lần với key | `(d % 2, -d)` | `O(n log n)` | `O(n)` | học thủ thuật `key` tuple |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `print(sorted([4, 8, 6], reverse=True), [d % 2 for d in [1, 2, 3]])` in ra gì?

**Bài 2 (tính tay).** `num = 247`. Hãy tìm số lớn nhất sau các lần hoán đổi cùng parity.

**Bài 3 (điền TODO).** Hoàn thành hàm trả về số **nhỏ nhất** bằng hoán đổi cùng parity.

In [22]:
def smallest_by_parity(num):
    # TODO: giống bài chính nhưng sắp xếp tăng dần từng nhóm
    pass

# chưa gọi hàm ở đây để notebook chạy không lỗi

**Bài 4 (tự viết lại).** Viết `largest_by_mod3(num)`: được hoán đổi hai chữ số có **cùng số dư khi chia cho 3**; trả về số lớn nhất.

In [23]:
def largest_by_mod3(num):
    # TODO: nhóm chữ số theo d % 3 (ba nhóm), sắp xếp giảm dần mỗi nhóm rồi điền lại
    pass

**Bài 5 (mini challenge).** Viết `count_reachable(num)` trả về **số giá trị khác nhau** có thể đạt được bằng hoán đổi cùng parity. Gợi ý: với mỗi nhóm, số hoán vị phân biệt của một đa tập là `n! / (c1! · c2! · ...)`, và hai nhóm độc lập nên nhân với nhau.

In [24]:
def count_reachable(num):
    # TODO: dùng math.factorial và Counter
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `sorted([4,8,6], reverse=True) = [8,6,4]`; parity của `1,2,3` là `1,0,1`. Bài 2: hãy phân loại `2` (chẵn), `4` (chẵn), `7` (lẻ).
- **Mức 2 (vừa):** Bài 2: nhóm chẵn `{2,4}` ở vị trí 0, 1; nhóm lẻ `{7}` ở vị trí 2 → số lớn nhất là `427`. Bài 3: đổi `reverse=True` thành tăng dần.
- **Mức 3 (sâu):** Bài 4: `groups[d % 3]`, giữ con trỏ cho mỗi nhóm. Bài 5: dùng `Counter` mỗi nhóm → `math.factorial(len) // prod(factorial(c))`.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [25]:
import math
from collections import Counter

# Bài 1 và 2
assert (sorted([4, 8, 6], reverse=True), [d % 2 for d in [1, 2, 3]]) == ([8, 6, 4], [1, 0, 1])
assert Solution().largestInteger(247) == 427

# Bài 3
def smallest_by_parity(num):
    d = [int(c) for c in str(num)]
    ev = sorted(x for x in d if x % 2 == 0)
    od = sorted(x for x in d if x % 2 == 1)
    ei = oi = 0
    out = []
    for x in d:
        if x % 2 == 0:
            out.append(ev[ei]); ei += 1
        else:
            out.append(od[oi]); oi += 1
    return int("".join(map(str, out)))

# Bài 4
def largest_by_mod3(num):
    d = [int(c) for c in str(num)]
    groups = {r: sorted((x for x in d if x % 3 == r), reverse=True) for r in range(3)}
    ptr = {0: 0, 1: 0, 2: 0}
    ans = 0
    for x in d:
        r = x % 3
        ans = ans * 10 + groups[r][ptr[r]]
        ptr[r] += 1
    return ans

# Bài 5
def count_reachable(num):
    d = [int(c) for c in str(num)]
    total = 1
    for group in ([x for x in d if x % 2 == 0], [x for x in d if x % 2 == 1]):
        ways = math.factorial(len(group))
        for c in Counter(group).values():
            ways //= math.factorial(c)
        total *= ways
    return total

assert smallest_by_parity(65875) == 65857
assert smallest_by_parity(1234) == 1234
assert smallest_by_parity(4321) == 2143
assert largest_by_mod3(1234) == 4231        # nhóm dư 1: {1,4} -> 4,1; dư 2: {2}; dư 0: {3}
assert largest_by_mod3(333) == 333
assert largest_by_mod3(123) == 123          # ba chữ số thuộc ba nhóm khác nhau
assert count_reachable(1234) == 4          # chẵn {2,4}: 2 cách, lẻ {1,3}: 2 cách
assert count_reachable(65875) == 2 * 3      # chẵn {6,8}: 2, lẻ {5,7,5}: 3
assert count_reachable(7) == 1
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (tách hai nhóm parity, sắp xếp giảm dần, điền lại theo parity gốc)

```python
class Solution:
    def largestInteger(self, num: int) -> int:
        digits = [int(c) for c in str(num)]
        evens = sorted((d for d in digits if d % 2 == 0), reverse=True)
        odds = sorted((d for d in digits if d % 2 == 1), reverse=True)
        ei = oi = 0
        ans = 0
        for d in digits:
            if d % 2 == 0:
                ans = ans * 10 + evens[ei]; ei += 1
            else:
                ans = ans * 10 + odds[oi]; oi += 1
        return ans
```

**Checklist**

- [ ] Hai nhóm chẵn/lẻ được xử lý **độc lập**, không trộn lẫn?
- [ ] Mỗi nhóm sắp xếp **giảm dần** (số lớn nhất)?
- [ ] Điền lại theo **parity của chữ số gốc tại vị trí đó**?
- [ ] Đã thử một chữ số, toàn chẵn, toàn lẻ, và số có `0`?
- [ ] Đã tránh `remove`/`pop(0)` không cần thiết?
- [ ] Signature đúng `largestInteger(self, num: int) -> int` và có `from typing import List`?